# Diffusion Tinker - DDRL Demo

Train SD3.5-Medium with DDRL (Data-regularized RL) to improve aesthetic quality.

**Requirements:** Colab Pro with A100 or T4 GPU. ~16GB VRAM minimum.

In [ ]:
# Install diffusion-tinker
!pip install -q git+https://github.com/srijitiyer/diffusion-tinker.git
!pip install -q open-clip-torch wandb

In [ ]:
# Login to HuggingFace (SD3.5 requires license acceptance)
# Accept at: https://huggingface.co/stabilityai/stable-diffusion-3.5-medium
from huggingface_hub import login
login()

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
from diffusion_tinker import DDRLTrainer, DDRLConfig

prompts = [
    "a photograph of a mountain landscape at golden hour",
    "a portrait of a cat sitting on a windowsill",
    "an oil painting of a city street in the rain",
    "a macro photograph of a flower with morning dew",
    "a watercolor painting of a sailboat on calm water",
    "a photograph of a cozy library with warm lighting",
    "an illustration of a forest path in autumn",
    "a photograph of ocean waves crashing on rocks",
]

# T4: use smaller batch, lower resolution
# A100: can increase num_samples_per_prompt to 8
is_t4 = "T4" in torch.cuda.get_device_name(0)

config = DDRLConfig(
    data_beta=0.01,
    num_samples_per_prompt=2 if is_t4 else 4,
    num_inference_steps=10,
    num_eval_inference_steps=28,
    guidance_scale=7.0,
    noise_level=0.7,
    resolution=512,
    learning_rate=1e-4,
    lora_rank=16 if is_t4 else 32,
    lora_alpha=32 if is_t4 else 64,
    num_epochs=30,
    save_every=10,
    eval_every=5,
    output_dir="./ddrl_output",
    gradient_checkpointing=True,
    mixed_precision="bf16",
)

trainer = DDRLTrainer(
    model="stabilityai/stable-diffusion-3.5-medium",
    reward_funcs="aesthetic",
    train_prompts=prompts,
    config=config,
)

trainer.train()

In [ ]:
# View eval samples
from pathlib import Path
from IPython.display import display, Image as IPImage

eval_dirs = sorted(Path("./ddrl_output").glob("eval-*"))
if eval_dirs:
    latest = eval_dirs[-1]
    print(f"Showing samples from {latest.name}:")
    for img_path in sorted(latest.glob("*.png")):
        print(img_path.name)
        display(IPImage(filename=str(img_path), width=256))